# 02_core_functions
Every function the main notebooks call lives here. It has three parts:

1. **Common**: reading the control table, logging, and the copy and unzip steps. Both pipelines use these.
2. **NPHC** (monthly): saves one metadata row per JSON file. These functions start with `nphc_`.
3. **HSAR** (daily): loads each dated folder with Auto Loader. These functions start with `hsar_`.

# 1. Common (both pipelines)

In [0]:
def get_active_control_entries(src_system):
    return spark.sql(f"""
        SELECT s_no, src_system, zip_source, zip_source_path, zip_tgt_path, unzip_tgt_path,
               tgt_catalog, tgt_schema, tgt_table
        FROM {control_table}
        WHERE is_active = 1 AND src_system = '{src_system}'
        ORDER BY s_no
    """).collect()


def write_logs(log_records, run_id):
    if not log_records:
        return
    rows = [
        (run_id, r.get("stage"),
         int(r["s_no"]) if r.get("s_no") is not None else None,
         r.get("source"), r.get("file_name"), r.get("source_path"), r.get("target_path"),
         r.get("status"), r.get("message"), r.get("start_time"), r.get("end_time"))
        for r in log_records
    ]
    spark.createDataFrame(rows, schema=LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(log_table)

## Copy and unzip
Both sources name their zips the same way. The **MID date** in the name decides the folder (`raw/<mid>/` and `processed/<mid>/`).

Deciding what to copy is simple: list everything at the source and copy the zips that aren't in `raw/` yet. The first run copies all of them. If a day or month gets missed, its zip is still missing from `raw/`, so the next run picks it up.

Want to reload a zip? Delete its copy from `raw/` and run again.

In [0]:
def parse_file_name(file_name):
    # 'case_data_001_20220605_20220620_20260101_235954.zip'
    #  -> {'table_name': 'case_data_001', 'start_date': '20220605', 'mid_date': '20220620', 'end_date': '20260101'}
    # gives back None when the name doesn't match the pattern
    m = FILE_NAME_PATTERN.match(file_name)
    return m.groupdict() if m else None


def zip_folder_date(file_name):
    # the folder name used in raw/ and processed/ (the MID date), or None if the name doesn't match
    parts = parse_file_name(file_name)
    return parts["mid_date"] if parts else None


def nphc_months(zip_files):
    # months (yyyymm) taken from the END dates, e.g. ['202604', '202605']
    return sorted({d[:6] for d in zip_end_dates(zip_files)})


def zip_end_dates(zip_files):
    # the END dates (yyyymmdd) of these zips, sorted. The preview uses them to check for a new month or day
    return sorted({p["end_date"] for p in (parse_file_name(f.name) for f in zip_files) if p})


def list_zip_files(source_path):
    source_path = source_path.rstrip("/") + "/"
    return [f for f in dbutils.fs.ls(source_path) if not f.isDir() and f.name.lower().endswith(".zip")]


def raw_zip_path(file_name, target_base_path):
    # where the zip should sit in raw/: <zip_tgt_path>/<mid_date>/<file_name> (None if the name doesn't match)
    folder_date = zip_folder_date(file_name)
    if folder_date is None:
        return None
    return f"{target_base_path.rstrip('/')}/{folder_date}/{file_name}"


def select_zips_to_copy(zip_files, ctl):
    # keep only the zips that aren't in raw/ yet
    new = []
    for f in zip_files:
        raw_path = raw_zip_path(f.name, ctl["zip_tgt_path"])
        if raw_path is None or not os.path.exists(raw_path):      # odd names are kept on purpose so they show up as SKIPPED in the log
            new.append(f)
    return new


def discard_raw_copy(path):
    # if unzip or load failed, delete the raw copy so the next run sees the zip as new and tries again
    try:
        dbutils.fs.rm(path)
    except Exception as e:
        print(f"WARNING: could not remove {path}: {e}")


def copy_zip(file_info, ctl):
    # copies one zip into raw/<date>/ and returns a log record
    start_time = datetime.now()
    record = {"s_no": ctl["s_no"], "stage": "COPY", "source": ctl["src_system"], "file_name": file_info.name,
              "source_path": file_info.path, "target_path": None, "start_time": start_time}

    target_file = raw_zip_path(file_info.name, ctl["zip_tgt_path"])
    if target_file is None:
        record.update(status="SKIPPED", message="File name does not match <name>_<yyyymmdd>_<yyyymmdd>_<yyyymmdd>_<hhmmss>.zip",
                      end_time=datetime.now())
        return record

    record["target_path"] = target_file
    try:
        dbutils.fs.mkdirs(target_file.rsplit("/", 1)[0] + "/")
        dbutils.fs.cp(file_info.path, target_file)
        record.update(status="SUCCESS", message="Copied successfully")
    except Exception as e:
        record.update(status="FAILED", message=str(e))
    record["end_time"] = datetime.now()
    return record


def unzip_zip(copy_record, unzip_base_path):
    # unzips one copied zip into <unzip_tgt_path>/<date>/<zip_name>/
    # every zip gets its own sub-folder, so files from different zips can't overwrite each other
    start_time = datetime.now()
    file_name = copy_record["file_name"]
    zip_path  = copy_record["target_path"]
    record = {"s_no": copy_record["s_no"], "stage": "UNZIP", "source": copy_record["source"],
              "file_name": file_name, "source_path": zip_path, "target_path": None, "start_time": start_time}

    folder_date   = zip_folder_date(file_name)
    target_folder = f"{unzip_base_path.rstrip('/')}/{folder_date}/{file_name.rsplit('.', 1)[0]}/"
    record["target_path"] = target_folder
    try:
        dbutils.fs.mkdirs(target_folder)
        with zipfile.ZipFile(zip_path.replace("dbfs:", ""), "r") as z:
            z.extractall(target_folder)
        record.update(status="SUCCESS", message="Extracted successfully")
    except zipfile.BadZipFile:
        record.update(status="FAILED", message="Invalid ZIP file")
    except Exception as e:
        record.update(status="FAILED", message=str(e))
    record["end_time"] = datetime.now()
    return record


def copy_and_unzip(run_id, control_entries, log_records):
    # does the copy and unzip steps for the given control rows, adds to log_records and writes the logs
    ctl_by_sno = {c["s_no"]: c for c in control_entries}

    print("=== Step 1: copy ===")
    for ctl in control_entries:
        all_zips = list_zip_files(ctl["zip_source_path"])
        zips = select_zips_to_copy(all_zips, ctl)
        print(f"s_no={ctl['s_no']} | {ctl['zip_source_path']}: {len(all_zips)} zip file(s) at source, {len(zips)} to copy")
        for f in zips:
            rec = copy_zip(f, ctl)
            log_records.append(rec)
            print(f"  [COPY:{rec['status']}] {rec['file_name']}: {rec['message']}")

    print("=== Step 2: unzip ===")
    copied = [r for r in log_records if r["stage"] == "COPY" and r["status"] == "SUCCESS"]
    print(f"{len(copied)} zip file(s) to unzip")
    for copy_rec in copied:
        rec = unzip_zip(copy_rec, ctl_by_sno[copy_rec["s_no"]]["unzip_tgt_path"])
        log_records.append(rec)
        print(f"  [UNZIP:{rec['status']}] {rec['file_name']}: {rec['message']}")
        if rec["status"] == "FAILED":
            discard_raw_copy(copy_rec["target_path"])        # so the next run tries this zip again

    write_logs(log_records, run_id)
    print(f"Logged {len(log_records)} COPY/UNZIP record(s) to {log_table}")

# 2. NPHC (monthly): ingest the JSON files

In [0]:
def nphc_create_target_table(target):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target} (
        src_system      STRING,
        file_id         BIGINT,      -- serial no.
        zip_file        STRING,      -- zip the JSON file came from
        file_name       STRING,
        file_date       TIMESTAMP,   -- modified time of the extracted file
        file_path       STRING,
        dt_datafile     STRING,      -- middle date from file name (yyyymmdd)
        file_size_mb    DOUBLE,
        file_structure  VARIANT,
        table_name      STRING,      -- from file name
        created_by      STRING,
        created_on      TIMESTAMP
    ) USING DELTA
    """)
    # table made by an older version without zip_file? add the column
    if "zip_file" not in spark.table(target).columns:
        spark.sql(f"ALTER TABLE {target} ADD COLUMNS (zip_file STRING AFTER file_id)")


def nphc_list_json_files(path):
    # every .json file under this path, sub-folders included
    out = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            out.extend(nphc_list_json_files(item.path))
        elif item.name.lower().endswith(".json"):
            out.append(item)
    return sorted(out, key=lambda x: x.path)


def nphc_log_load_event(run_id, ctl, zip_file, f, target, status, started, error=None):
    # adds one row to the load log. A logging problem should never stop the load, so errors are only printed
    try:
        row = (
            run_id, ctl["src_system"], ctl["unzip_tgt_path"], zip_file,
            f.name if f else None, f.path if f else None,
            f.size / (1024.0 * 1024.0) if f else None,
            int(f.modificationTime) if f else None,
            target, status,
            type(error).__name__ if error else None,
            (str(error)[:4000] + "\n" + traceback.format_exc()[-2000:]) if error else None,
            started, datetime.now(), EXECUTED_BY,
        )
        spark.createDataFrame([row], LOAD_LOG_SCHEMA).write.mode("append").saveAsTable(load_log_table)
    except Exception as log_err:
        print(f"WARNING: could not write load log row: {log_err}")


def nphc_load_file(f, target, zip_file, src_system):
    # reads one JSON file, grabs its structure and saves one metadata row to the target
    parts = parse_file_name(f.name)
    if parts is None:
        raise ValueError(f"Bad file name '{f.name}'. Expected <table>_<yyyymmdd>_<yyyymmdd>_<yyyymmdd>_<hhmmss>.json")
    dt_datafile = datetime.strptime(parts["mid_date"], "%Y%m%d").strftime("%Y%m%d")

    file_structure = spark.read.option("multiLine", "true").json(f.path).schema.json()

    # remove the row from an earlier run of this file first, so re-runs don't create duplicates
    spark.sql(f"DELETE FROM {target} WHERE file_path = :p", args={"p": f.path})
    file_id = spark.sql(f"SELECT COALESCE(MAX(file_id), 0) + 1 FROM {target}").first()[0]

    (spark.createDataFrame([Row(
            src_system     = src_system,
            file_id        = file_id,
            zip_file       = zip_file,
            file_name      = f.name,
            file_date      = datetime.fromtimestamp(f.modificationTime / 1000),
            file_path      = f.path,
            dt_datafile    = dt_datafile,
            file_size_mb   = f.size / (1024.0 * 1024.0),
            file_structure = file_structure,
            table_name     = parts["table_name"],
            created_by     = EXECUTED_BY,
        )])
        .withColumn("file_structure", F.parse_json("file_structure"))
        .withColumn("created_on", F.current_timestamp())
        .select("src_system", "file_id", "zip_file", "file_name", "file_date", "file_path", "dt_datafile",
                "file_size_mb", "file_structure", "table_name", "created_by", "created_on")
        .write.mode("append").saveAsTable(target))


def nphc_ingest_zip(unzip_record, ctl, run_id):
    # loads every JSON file that came out of one zip and returns the counts, e.g. {'SUCCESS': 3, 'FAILED': 0}
    target   = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    zip_file = unzip_record["file_name"]
    counts   = {"SUCCESS": 0, "FAILED": 0}

    started = datetime.now()
    try:
        nphc_create_target_table(target)
        files = nphc_list_json_files(unzip_record["target_path"])
        if not files:
            raise ValueError(f"No .json files found in {unzip_record['target_path']}")
    except Exception as e:
        nphc_log_load_event(run_id, ctl, zip_file, None, target, "FAILED", started, e)
        counts["FAILED"] += 1
        print(f"  [INGEST:FAILED] {zip_file}: {str(e)[:300]}")
        return counts

    for f in files:
        started = datetime.now()
        try:
            nphc_load_file(f, target, zip_file, ctl["src_system"])
            nphc_log_load_event(run_id, ctl, zip_file, f, target, "SUCCESS", started)
            counts["SUCCESS"] += 1
            print(f"  [INGEST:SUCCESS] {f.name}")
        except Exception as e:
            nphc_log_load_event(run_id, ctl, zip_file, f, target, "FAILED", started, e)
            counts["FAILED"] += 1
            print(f"  [INGEST:FAILED] {f.name}: {str(e)[:300]}")
    return counts

# 3. HSAR (daily): Auto Loader

Once the zips are unzipped, each dated folder (`<unzip_tgt_path>/<mid_date>/`) is loaded with Auto Loader. Every JSON file becomes one row, holding the raw text and a parsed `VARIANT` copy. Each folder has its own checkpoint at `<unzip_tgt_path>/checkpoint/<yyyymmdd>`.

Only folders that got new zips in this run are loaded. If another zip lands in a day that was already loaded, the checkpoint means Auto Loader only reads the new files.

If a folder fails to load, `hsar_reset_folder` clears its checkpoint and any half-loaded rows, so the retry starts clean.

In [0]:
HSAR_TABLE_NAME_REGEX = r"^(.+?)(?:_\d{8}){1,3}_\d{6}\.[A-Za-z]+$"     # pulls the table name out of <table_name>_<dates>_<hhmmss>.json


def hsar_checkpoint(ctl, dated_folder):
    return f"{ctl['unzip_tgt_path'].rstrip('/')}/checkpoint/{dated_folder}"


def hsar_folder_path(ctl, dated_folder):
    return f"{ctl['unzip_tgt_path'].rstrip('/')}/{dated_folder}/"


def hsar_reset_folder(ctl, dated_folder):
    # wipes a dated folder from the target: its checkpoint and any rows already loaded from it
    target = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    dbutils.fs.rm(hsar_checkpoint(ctl, dated_folder), True)
    if spark.catalog.tableExists(target):
        spark.sql(f"DELETE FROM {target} WHERE file_path = '{hsar_folder_path(ctl, dated_folder)}'")


def hsar_load_folder(ctl, dated_folder):
    # loads one dated folder from processed/ into the target table and returns how many of its files are in the target
    from pyspark.sql.window import Window

    target      = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    folder_path = hsar_folder_path(ctl, dated_folder)

    m          = F.col("_metadata")
    name_dt    = F.regexp_extract(m["file_name"], HSAR_FILE_DATE_PATTERN, 1)
    dt_file    = F.when(name_dt != "", name_dt).otherwise(None)
    tbl        = F.regexp_extract(m["file_name"], HSAR_TABLE_NAME_REGEX, 1)
    table_name = F.when(tbl != "", tbl).otherwise(None)
    # paths look like <processed>/<yyyymmdd>/<zip_name>/<file>.json, so the zip name can be read from the path
    zip_file = F.concat(F.regexp_extract(m["file_path"], r"/\d{8}/([^/]+)/", 1), F.lit(".zip"))

    raw = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "text")
        .option("wholetext", "true")
        .option("pathGlobFilter", "*.json")
        .option("cloudFiles.maxFilesPerTrigger", HSAR_MAX_FILES_PER_TRIGGER)
        .load(folder_path)
    )

    df = (
        raw.select(
            F.lit(ctl["src_system"]).alias("src_system"),
            zip_file.alias("zip_file"),
            m["file_name"].alias("file_name"),
            m["file_modification_time"].alias("file_date"),
            F.lit(folder_path).alias("file_path"),
            dt_file.alias("dt_data_file"),
            (m["file_size"] / 1024.0).alias("file_size_kb"),
            F.col("value").alias("json_original_data"),
            F.expr("try_parse_json(value)").alias("json_data_for_parsing"),
            table_name.alias("table_name"),
        )
        .withColumn("dt_data_file_date", F.to_date("dt_data_file", "yyyyMMdd"))
        .withColumn("data_year", F.year("dt_data_file_date"))
        .withColumn("data_month", F.month("dt_data_file_date"))
        .withColumn("created_by", F.lit(EXECUTED_BY))
        .withColumn("created_on", F.current_timestamp())
    )

    def write_batch(batch_df, batch_id):
        # give each file a running file_id, then append the batch to the target
        if batch_df.isEmpty():
            return
        last_id = spark.sql(f"SELECT COALESCE(MAX(file_id), 0) FROM {target}").first()[0] if spark.catalog.tableExists(target) else 0
        numbered = batch_df.withColumn(
            "file_id", (F.row_number().over(Window.orderBy("zip_file", "file_name")) + F.lit(last_id)).cast("long")
        )
        (numbered.select("src_system", "file_id", "zip_file", "file_name", "file_date", "file_path", "dt_data_file",
                         "file_size_kb", "json_original_data", "json_data_for_parsing", "table_name",
                         "dt_data_file_date", "data_year", "data_month", "created_by", "created_on")
                 .write.format("delta").mode("append").partitionBy("data_year", "data_month").saveAsTable(target))

    q = (
        df.writeStream
        .foreachBatch(write_batch)
        .option("checkpointLocation", hsar_checkpoint(ctl, dated_folder))
        .trigger(availableNow=True)
        .start()
    )
    q.awaitTermination()

    return spark.table(target).filter(F.col("file_path") == folder_path).count()